In [ ]:
import matplotlib.pyplot as plt

import torch

from occupycy.data import OccuData
from occupycy.models import LinearMLPOccuModel, LinearOccuModel, ConstantOccuModel
from occupycy.fit import fit_sgd_like, fit_lbfgs_full_batch
from occupycy.likelihood import nll_logits

from occupycy.data.simulate_data import simulate_linear, simulate_non_linear

seed = 123

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device = torch.device("cpu")

# Simulate data (non-linear or linear)

In [ ]:
# SIMULATE NON-LINEAR DATA
# Choose one model for PSI and one for D. 

n_cov_psi = 2
n_cov_d = 1

# # * * * Example constant PSI * * *
# def true_f_psi(X_psi):
#     psi = torch.tensor([0.3, 0.5], device=X_psi.device, dtype=X_psi.dtype) # shape: (n_species,)
#     return psi.unsqueeze(0).expand(X_psi.shape[0], -1) # shape: (n_sites, n_species)

# # * * * Example linear PSI * * *
# def true_f_psi(X_psi):
#     psi = torch.sigmoid(
#         torch.tensor([0.0, -1.0]) # shape: (n_species,) intercept
#         + X_psi @ torch.tensor([[0.0, 1.0], [-1.0, 0.5]]).T # shape: (n_sites, n_cov_psi) @ (n_cov_psi, n_species) -> (n_sites, n_species)
#     ) # shape: from X_psi of shape (n_sites, n_cov_psi) to psi of shape (n_sites, n_species)
#     return psi

# * * * Example non-linear PSI * * *
def true_f_psi(X_psi):
    x1 = X_psi[..., :1] # shape: (n_sites, 1)
    logit1 = x1 @ torch.tensor([[-1.0], [0.0]], device=X_psi.device, dtype=X_psi.dtype).T # shape: (n_sites, n_species)
    x2 = X_psi[..., 1] # shape: (n_sites,)
    mu = torch.tensor([0.3, -0.2], device=X_psi.device, dtype=X_psi.dtype)  # optima
    sigma = torch.tensor([0.5, 0.4], device=X_psi.device, dtype=X_psi.dtype)  # niche width
    logit2 = torch.stack([
        -((x2 - mu[0])**2 / sigma[0]**2),
        -((x2 - mu[1])**2 / sigma[1]**2),
    ], dim=-1) # shape: (n_sites, n_species)
    eps = 1e-8
    logit2 = logit2.clamp(torch.logit(torch.tensor(eps)), torch.logit(torch.tensor(1-eps)))
    return torch.sigmoid(logit1 + logit2)

# # * * * Example constant D * * *
# def true_f_d(X_d):
#     d = torch.tensor([0.7, 0.1], device=X_d.device, dtype=X_d.dtype) # shape: (n_species,)
#     return d.unsqueeze(0).expand(X_d.shape[0], -1) # shape: (n_sites, n_species)

# * * * Example linear D * * *
def true_f_d(X_d):
    d = torch.sigmoid(
        torch.tensor([-1.0, 0.5]) 
        + X_d @ torch.tensor([[-1.0], [-2.0]]).T
    )
    return d

# * * * Example Efforts * * *
def f_effort(x): # highly imbalanced effort
    return torch.exp(3*x)
# def f_effort(x): # moderately imbalanced effort
#     return 10 * x ** 2
# def f_effort(x): # constant effort
#     return torch.ones_like(x) * 100

y, visit_site_ids, site_ids_in_X_psi, X_psi, X_d, true_z = simulate_non_linear(
    n_sites=1000, 
    n_total_visits_max=int(1e6),
    n_cov_psi=n_cov_psi,
    n_cov_d=n_cov_d,
    f_effort=f_effort,
    f_psi=true_f_psi,
    f_d=true_f_d,
    seed=seed,
    plot_summary=True,
    verbose=True
)

eps = 0.0 # Small value to avoid logit issues with 0 and 1
def f_true_logit_psi(X_psi, eps=eps):
    return torch.logit(true_f_psi(X_psi).clamp(eps, 1-eps))
def f_true_logit_d(X_d, eps=eps):
    return torch.logit(true_f_d(X_d).clamp(eps, 1-eps))

In [ ]:
# # SIMULATE LINEAR DATA

# true_intercept_psi = torch.tensor([0.0, -1.0])                      # shape: (n_species,)
# true_coefs_psi = torch.tensor([[0.0, 1.0, 2.0], [-1.0, 0.5, -0.5]]) # shape: (n_species, n_cov_psi)
# true_intercept_d = torch.tensor([-1.0, 0.5])                        # shape: (n_species,)
# true_coefs_d = torch.tensor([[0.0], [1.0]])                         # shape: (n_species, n_cov_d)

# y, visit_site_ids, site_ids_in_X_psi, X_psi, X_d, true_z = simulate_linear(
#     n_sites=1000, 
#     n_total_visits_max=int(1e6),
#     # f_effort=lambda x1: torch.exp(3*x1),
#     # f_effort=lambda x1: 10 * x1 ** 2,
#     f_effort=lambda x1: torch.ones_like(x1) * 10,
#     intercept_psi=true_intercept_psi, 
#     coefs_psi=true_coefs_psi, 
#     intercept_d=true_intercept_d, 
#     coefs_d=true_coefs_d, 
#     seed=seed
# )

# def f_true_logit_psi(X_psi):
#     return true_intercept_psi + X_psi @ true_coefs_psi.T
# def f_true_logit_d(X_d):
#     return true_intercept_d + X_d @ true_coefs_d.T

In [ ]:
# Give a name to each site id (eg: site_000, site_001, ...)
fmt = f"site_{{:0{len(str(max(visit_site_ids)))}d}}"
visit_site_ids = [fmt.format(i) for i in visit_site_ids]
site_ids_in_X_psi = [fmt.format(i) for i in site_ids_in_X_psi]

# Fit

In [ ]:
# CREATE THE DATA OBJECT

data = OccuData(
    y=y, 
    visit_site_ids=visit_site_ids, 
    X_psi=X_psi, 
    X_d=X_d,
    site_ids_in_X_psi=site_ids_in_X_psi
)

In [ ]:
# SPLIT THE DATA

fraction_val, fraction_test = 0.15, 0.15
# fraction_val, fraction_test = 0.0, 0.0

n_val = int(data.n_sites * fraction_val)
n_test = int(data.n_sites * fraction_test)
n_train = data.n_sites - n_val - n_test
all_site_ids = sorted(list(set(visit_site_ids)))
sites_idxs = torch.randperm(data.n_sites)
train_ids = [all_site_ids[i] for i in sites_idxs[:n_train].tolist()]
val_ids = [all_site_ids[i] for i in sites_idxs[n_train:n_train+n_val].tolist()]
test_ids = [all_site_ids[i] for i in sites_idxs[n_train+n_val:].tolist()]
splitting={'train': train_ids, 'val': val_ids, 'test': test_ids}

In [ ]:
# DEFINE THE MODEL (linear + mlp, or linear, or constant)

model = LinearMLPOccuModel(
    n_species=y.shape[1], 
    n_cov_psi_linear=1, 
    n_cov_psi_mlp=1,
    n_cov_d_linear=1, 
    n_cov_d_mlp=0,
    hidden_dims_psi=[32, 32, 16], 
    hidden_dims_d=[],
    activation_fn_psi=torch.nn.ReLU, 
    activation_fn_d=torch.nn.ReLU,
    dropout_psi=0.0, 
    # dropout_psi=0.2, 
    dropout_d=0.0
)

# model = LinearOccuModel(
#     n_species=y.shape[1], 
#     n_cov_psi=X_psi.shape[1], 
#     n_cov_d=X_d.shape[1]
# )

# model = ConstantOccuModel(n_species=data.n_species)

In [ ]:
# FIT THE MODEL (SGD-like or LBFGS)

# --- SGD-like ---
res = fit_sgd_like(
    data, 
    model, 
    n_epochs=1000, 
    batch_size=200, 
    lr_psi=1e-3, 
    lr_d=1e-3,
    optimizer_psi_cls=torch.optim.SGD, 
    optimizer_d_cls=torch.optim.SGD,
    seed_dataloader=seed,
    splitting=splitting,
    device=device,
    patience_early_stopping=15
)
# --- end of SGD-like ---


# # --- LBFGS ---
# # Create the split on data object
# data.build_dataloaders(
#     batch_size=len(splitting['train']),
#     splitting=splitting,
#     shuffle=False,
#     device=device
# )

# # data_train = next(iter(data.train_loader))
# data_train = data.filtered_sites(site_id_mask=splitting['train'])

# res = fit_lbfgs_full_batch(
#     data_train, model, max_iter=1000, 
#     # lr_psi=0.1, lr_d=0.1, 
#     lr=5e-3,
#     device=device,
#     tolerance_grad=1e-8, tolerance_change=1e-10, history_size=100, line_search_fn="strong_wolfe"
# )
# # --- end of LBFGS ---

# Analyze outputs

In [ ]:
# EVALUATE THE MODEL
# Plot predicted vs true values AND compute NLL for each split

colors = {
    "train_pred": "#650ab9",
    "train_true": "#e377c2",
    "val_pred": "#d62728",
    "val_true": "#ff7f0e",
    "test_pred": "#0a92f3",
    "test_true": "#2ca02c",
}

n_species = y.shape[1]

true_nll, final_pred_nll = {}, {}
print("NLL for each split:")

fig, axes = plt.subplots(n_species, 2, figsize=(6, 3 * n_species))
min_range_psi, max_range_psi = torch.tensor([torch.inf] * n_species), torch.tensor([-torch.inf] * n_species)
min_range_d, max_range_d = torch.tensor([torch.inf] * n_species), torch.tensor([-torch.inf] * n_species)

for split in ['train', 'val', 'test']:
    
    if split == 'train':
        dl = data.train_loader if data.train_loader is not None else [data]
    else:
        dl = data.val_loader if split == 'val' else data.test_loader
        if dl is None:
            continue

    # Compute predicted and true logits for the whole split
    logit_psi_true = torch.tensor([], dtype=torch.float32)
    logit_d_true = torch.tensor([], dtype=torch.float32)
    logit_psi_pred = torch.tensor([], dtype=torch.float32)
    logit_d_pred = torch.tensor([], dtype=torch.float32)
    
    this_true_nll = 0.0
    this_pred_nll = 0.0
    for batch in dl:
        model.eval()
        with torch.no_grad():
            tmp_logit_psi_pred, tmp_logit_d_pred = model.forward_logits(
                X_psi=batch["X_psi"], X_d=batch["X_d"]
            )
        tmp_logit_psi_true = f_true_logit_psi(batch["X_psi"])
        tmp_logit_d_true = f_true_logit_d(batch["X_d"])
        # Append to the full tensors for plotting
        logit_psi_pred = torch.cat([logit_psi_pred, tmp_logit_psi_pred.cpu()])
        logit_d_pred = torch.cat([logit_d_pred, tmp_logit_d_pred.cpu()])
        logit_psi_true = torch.cat([logit_psi_true, tmp_logit_psi_true.cpu()])
        logit_d_true = torch.cat([logit_d_true, tmp_logit_d_true.cpu()])
        # Compute NLL for the split
        this_true_nll += nll_logits(
            y=batch["y"], logit_psi=tmp_logit_psi_true, logit_d=tmp_logit_d_true, 
            indices_visit_site=batch["indices_visit_site"]
        ).item()
        this_pred_nll += nll_logits(
            y=batch["y"], logit_psi=tmp_logit_psi_pred, logit_d=tmp_logit_d_pred, 
            indices_visit_site=batch["indices_visit_site"]
         ).item()
    true_nll[split] = this_true_nll
    final_pred_nll[split] = this_pred_nll
    print(f"TRUE:  {split:>6}: {true_nll[split]:>10.3f} (n_sites={len(splitting[split]):>5} "
          f"-> average nll={true_nll[split]/len(splitting[split]):.4f})")
    print(f"PRED:  {split:>6}: {final_pred_nll[split]:>10.3f} (n_sites={len(splitting[split]):>5} "
          f"-> average nll={final_pred_nll[split]/len(splitting[split]):.4f})")


    # Update ranges for plotting
    min_range_psi = torch.vstack([torch.vstack([logit_psi_true.min(dim=0).values, logit_psi_pred.min(dim=0).values]).max(dim=0).values, min_range_psi]).min(dim=0).values
    max_range_psi = torch.vstack([torch.vstack([logit_psi_true.max(dim=0).values, logit_psi_pred.max(dim=0).values]).min(dim=0).values, max_range_psi]).max(dim=0).values
    min_range_d = torch.vstack([torch.vstack([logit_d_true.min(dim=0).values, logit_d_pred.min(dim=0).values]).max(dim=0).values, min_range_d]).min(dim=0).values
    max_range_d = torch.vstack([torch.vstack([logit_d_true.max(dim=0).values, logit_d_pred.max(dim=0).values]).min(dim=0).values, max_range_d]).max(dim=0).values

    # Plot predicted vs true values
    for idx_sp in range(n_species):
        axes[idx_sp, 0].scatter(
            logit_psi_true[:, idx_sp].detach().cpu(), logit_psi_pred[:, idx_sp].detach().cpu(), 
            s=4, alpha=0.1, label=split, color=colors[f"{split}_pred"]
        )
        axes[idx_sp, 1].scatter(
            logit_d_true[:, idx_sp].detach().cpu(), logit_d_pred[:, idx_sp].detach().cpu(), 
            s=4, alpha=0.1, label=split, color=colors[f"{split}_pred"]
        )

for idx_sp in range(n_species):
    ax1, ax2 = axes[idx_sp, 0], axes[idx_sp, 1]
    
    ax1.plot(
        [min_range_psi[idx_sp].detach().cpu(), max_range_psi[idx_sp].detach().cpu()], 
        [min_range_psi[idx_sp].detach().cpu(), max_range_psi[idx_sp].detach().cpu()],
        linestyle="--", color="gray"
    )
    ax1.legend()
    ax1.set_title(f"psi - species {idx_sp}")
    ax1.set_xlabel("True psi")
    ax1.set_ylabel("Predicted psi")
    ax1.grid(True, which="both", ls="--")

    ax2.plot(
        [min_range_d[idx_sp].detach().cpu(), max_range_d[idx_sp].detach().cpu()], 
        [min_range_d[idx_sp].detach().cpu(), max_range_d[idx_sp].detach().cpu()], 
        linestyle="--", color="gray"
    )
    ax2.legend()
    ax2.set_title(f"d - species {idx_sp}")
    ax2.set_xlabel("True d")
    ax2.set_ylabel("Predicted d")
    ax2.grid(True, which="both", ls="--")

plt.tight_layout()
plt.show()


In [ ]:
def plot_this_loss_history(ax, split, res=res, final_pred_nll=final_pred_nll, 
                           true_nll=true_nll, splitting=splitting, 
                           colors=colors, epoch_max_range=5):
    if split == "val" and "val_loss" not in res.history:
        history = None
    else:
        history = torch.Tensor(res.history[f"{split}_loss"])
        history /= len(splitting[split])
    
    if history is not None:
        ax.plot(history, c=colors[f"{split}_pred"], # linestyle='o-', markersize=3,
                label=f"{split.capitalize()} history")
    ax.axhline(final_pred_nll[split] / len(splitting[split]), linestyle="dotted", c=colors[f"{split}_pred"],
               label=f"{split.capitalize()} final={final_pred_nll[split] / len(splitting[split]):.4f})")
    ax.axhline(true_nll[split] / len(splitting[split]), linestyle="--", c=colors[f"{split}_true"],
               label=f"{split.capitalize()} true={true_nll[split] / len(splitting[split]):.4f})")
    
    if history is not None:
        min_range = min(true_nll[split] / len(splitting[split]), history.min())
        max_range = max(true_nll[split] / len(splitting[split]), (history[epoch_max_range:]).max())
        ax.set_ylim(min_range - 0.1 * (max_range - min_range), max_range + 0.1 * (max_range - min_range))
    ax.set_yscale("log")

    ax.set_title(f"{split.capitalize()} Loss Curve")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (avg per site)")
    ax.grid(True, which="both", ls="--")
    
    ax.legend()

In [ ]:
# PLOT LOSS CURVES

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
plot_this_loss_history(axes[0], "train", epoch_max_range=min(50, len(res.history['train_loss'])//2))
plot_this_loss_history(axes[1], "val", epoch_max_range=min(50, len(res.history['val_loss'])//2))
plt.tight_layout()
plt.show()

In [ ]:
# PLOT LR HISTORIES

colors_lr = {
    "lr_psi": "#1f77b4",
    "lr_d": "#ff7f0e",
}

def plot_this_lr_history(ax, label_model, res=res):
    ax.plot(res.history[f"lr_{label_model}"], c=colors_lr[f"lr_{label_model}"], # linestyle='o-', markersize=3,
            label=f"LR {label_model}")
    ax.set_title(f"{label_model}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Learning Rate")
    ax.set_yscale("log")
    ax.grid(True, which="both", ls="--")
    ax.legend()

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
if "lr_psi" in res.history:
    plot_this_lr_history(axes[0], "psi")
if "lr_d" in res.history:
    plot_this_lr_history(axes[1], "d")
plt.suptitle("Learning Rate Histories")
plt.tight_layout()
plt.show()